In [8]:
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv
import os

load_dotenv()

True

# setting up the envirement variables

In [9]:
BIGQUERY_PROJECT = os.getenv('BIGQUERY_PROJECT')

# Connecting with bigquery to retrive the data from the tables

In [10]:
client = bigquery.Client(project=BIGQUERY_PROJECT)

# Extracting the results table from bigquery

In [11]:
fact_race_results_query = client.query("SELECT * FROM f1_data.fact_race_results")

# turning the query results into a dataframe

In [12]:
fact_race_results = fact_race_results_query.to_dataframe() 

In [17]:
fact_race_results.head()

,result_id,DriverId,TeamId,circuit_id,year,round_number,Position,GridPosition,Points,Laps,race_time_seconds,gap_to_winner_seconds,Status,ClassifiedPosition,is_classified,session_type
0,2020_16_89,aitken,williams,sakhir_grand_prix,2020,16,16,17,0,87,33.674,-5441.440,Finished,16,True,R
1,2019_12_23,albon,toro_rosso,hungarian_grand_prix,2019,12,10,12,1,69,NaN,NaN,+1 Lap,10,True,R
2,2020_10_23,albon,red_bull,russian_grand_prix,2020,10,10,15,1,53,97.860,-5542.504,Finished,10,True,R
3,2022_3_23,albon,williams,australian_grand_prix,2022,3,10,20,1,58,79.382,-5187.166,Finished,10,True,R
4,2022_14_23,albon,williams,belgian_grand_prix,2022,14,10,6,1,44,101.900,-5050.994,Finished,10,True,R


In [15]:
fact_race_results.isnull().sum()

result_id                   0
DriverId                    0
TeamId                      0
circuit_id                  0
year                        0
round_number                0
Position                    0
GridPosition                3
Points                      0
Laps                        0
race_time_seconds        1153
gap_to_winner_seconds    1153
Status                      0
ClassifiedPosition          0
is_classified               0
session_type                0
dtype: int64

# Extracting laps table from bigquery

In [18]:
fact_lap_times_query = client.query("SELECT * FROM f1_data.fact_lap_times")

In [19]:
fact_lap_times = fact_lap_times_query.to_dataframe()

In [20]:
fact_lap_times.head()

,lap_id,DriverId,circuit_id,year,round_number,lap_number,stint,lap_time_seconds,sector1_seconds,sector2_seconds,...,speedSt,compound,tyre_life,fresh_tyre,is_pit_lap,is_valid_lap,is_personal_best,Position,track_status,session_type
0,2018_21_ALO_24,alonso,abu_dhabi_grand_prix,2018,21,24,1,106.905,18.356,45.269,...,299.0,ULTRASOFT,24,True,False,True,False,10,1,R
1,2018_21_HAR_49,brendon_hartley,abu_dhabi_grand_prix,2018,21,49,2,105.099,18.187,44.757,...,300.0,SUPERSOFT,47,True,False,True,False,12,1,R
2,2018_21_SIR_48,sirotkin,abu_dhabi_grand_prix,2018,21,48,2,105.223,18.118,44.811,...,307.0,ULTRASOFT,13,True,False,True,False,15,21,R
3,2018_21_PER_45,perez,abu_dhabi_grand_prix,2018,21,45,2,103.983,18.071,43.955,...,307.0,SUPERSOFT,19,True,False,True,False,8,1,R
4,2018_21_VER_17,max_verstappen,abu_dhabi_grand_prix,2018,21,17,1,106.678,18.353,44.441,...,295.0,HYPERSOFT,20,False,True,False,False,2,1,R


In [21]:
fact_lap_times.isnull().sum()

lap_id                 0
DriverId               0
circuit_id             0
year                   0
round_number           0
lap_number             0
stint                646
lap_time_seconds    3156
sector1_seconds     3997
sector2_seconds      376
sector3_seconds      535
speedI1              283
speedI2             1170
speedFl             6435
speedSt              361
compound               0
tyre_life           1361
fresh_tyre             0
is_pit_lap             0
is_valid_lap           0
is_personal_best     210
Position             281
track_status           0
session_type           0
dtype: int64

# The goal that i'm trying to reach is building a model that can predict if a driver has been DNF or has finished the race, looking at the data, most of null rows in lap times belongs to drivers that has been DNF, most of features that i need are in the lap times table, so i have to join the result that have the target with lap times table that have the features

In [22]:
fact_race_results.columns

Index(['result_id', 'DriverId', 'TeamId', 'circuit_id', 'year', 'round_number',
       'Position', 'GridPosition', 'Points', 'Laps', 'race_time_seconds',
       'gap_to_winner_seconds', 'Status', 'ClassifiedPosition',
       'is_classified', 'session_type'],
      dtype='object')

In [23]:
fact_lap_times.columns

Index(['lap_id', 'DriverId', 'circuit_id', 'year', 'round_number',
       'lap_number', 'stint', 'lap_time_seconds', 'sector1_seconds',
       'sector2_seconds', 'sector3_seconds', 'speedI1', 'speedI2', 'speedFl',
       'speedSt', 'compound', 'tyre_life', 'fresh_tyre', 'is_pit_lap',
       'is_valid_lap', 'is_personal_best', 'Position', 'track_status',
       'session_type'],
      dtype='object')

# as we can see we can join the two tables using DriverId, then we have to aggregate lap times on race table level, bcs if we do the opposite will have each lap with a status of dnf or finished